# 2 · Vector layers

The vector builders draw a pyramids `FeatureCollection`: `points` (optionally coloured/sized by an
attribute), `path` (lines), `polygons` (outlines or fills) and `choropleth` (polygons coloured by a
column). We use the real **Rhine-basin** layers — 34 discharge gauges, the river centre-lines and the
basin outline (all EPSG:3035, reprojected to the display CRS by pyramids).

**Setup** — Bokeh extension and the three Rhine vector files.

In [ ]:
from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import holoviews as hv
hv.extension("bokeh")            # the interactive tier renders through Bokeh

from pyramids.feature import FeatureCollection
from digitalearth.interactive import InteractiveMap

gauges = FeatureCollection.read_file(str(DATA / "rhine_gauges.geojson"))
river  = FeatureCollection.read_file(str(DATA / "rhine_river_centerline.geojson"))
basin  = FeatureCollection.read_file(str(DATA / "rhine_basin.geojson"))

### `points` — coloured by an attribute
Colour the gauges by their `discharge` value and enlarge the markers. Hover is enabled by default, so each point shows its data in the browser.

In [ ]:
m = InteractiveMap(crs=3857, title="Rhine gauges by discharge")
m.points(gauges, value_column="discharge", size=10, cmap="plasma")
m

### `path` — line features
`path` draws line geometries — here the river centre-lines.

In [ ]:
m = InteractiveMap(crs=3857, title="Rhine river network")
m.path(river, color="navy")
m

### `polygons` — outlines
With no `column`, `polygons` draws outline-only (transparent fill) — good for showing an extent like the basin boundary.

In [ ]:
m = InteractiveMap(crs=3857, title="Rhine basin outline")
m.polygons(basin, line_color="firebrick")
m

### `choropleth` — polygons coloured by a column
`choropleth` fills polygons by an attribute. We build visible polygons by buffering each gauge into an 8 km 'catchment' circle (EPSG:3035 is in metres) and colour them by `discharge`. It needs the column name and adds a colorbar.

In [ ]:
catchments = gauges.copy()
catchments["geometry"] = catchments.geometry.buffer(8000.0)   # 8 km circles around each gauge

m = InteractiveMap(crs=3857, title="gauge catchments by discharge")
m.choropleth(catchments, "discharge", cmap="viridis")
m

Layers **compose**: basin outline, river network and gauges in one overlay — a finished basin map.

In [ ]:
m = InteractiveMap(crs=3857, title="Rhine basin overview")
m.polygons(basin, line_color="grey")
m.path(river, color="steelblue")
m.points(gauges, value_column="discharge", size=8, cmap="plasma")
m